# Bulk RNA-seq – Fibrosis vs Normal (PyDESeq2)

الخطوات: تحميل الـ counts ← فلترة ← metadata ← DESeq2 ← النتائج الدالة ← Volcano ← PCA ← Heatmap ← Gene annotation ← GO/KEGG enrichment ← GSEA

## 1. Install & imports

In [ ]:
!pip install -q pydeseq2 mygene gseapy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from sklearn.decomposition import PCA

## 2. Load counts\nعدّل `COUNTS_PATH` لمسار ملفك. المتوقع: عمود اسمه `Gene` + عمود لكل عينة (raw counts).

In [ ]:
COUNTS_PATH = "counts.csv"   # <-- عدّل هذا

df = pd.read_csv(COUNTS_PATH, sep=None, engine="python")
df.set_index("Gene", inplace=True)
df = df.round().astype(int)          # DESeq2 يحتاج integer counts
print(df.shape)
df.head()

## 3. Filter low-count genes\nنحتفظ بالجينات التي لها 10 قراءات أو أكثر في 3 عينات على الأقل.

In [ ]:
df_clean = df.copy()
df_filtered = df_clean[(df_clean >= 10).sum(axis=1) >= 3]
print(f"Genes before: {df_clean.shape[0]} | after: {df_filtered.shape[0]}")

## 4. Metadata\nنستخرج الـ condition من اسم العينة نفسه بدل كتابتها يدوياً، حتى لا يحصل عدم تطابق بين ترتيب العينات والـ labels.

In [ ]:
counts_for_dds = df_filtered.T       # DESeq2: العينات في الصفوف، الجينات في الأعمدة

metadata = pd.DataFrame(
    {"condition": ["fibrosis" if "fibrosis" in s.lower() else "normal" for s in counts_for_dds.index]},
    index=counts_for_dds.index,
)
metadata["condition"] = pd.Categorical(metadata["condition"], categories=["normal", "fibrosis"])
metadata

## 5. DESeq2

In [ ]:
try:
    dds = DeseqDataSet(counts=counts_for_dds, metadata=metadata, design="~condition")
except TypeError:   # إصدارات pydeseq2 القديمة
    dds = DeseqDataSet(counts=counts_for_dds, metadata=metadata, design_factors="condition")

dds.deseq2()

In [ ]:
stat_res = DeseqStats(dds, contrast=["condition", "fibrosis", "normal"])
stat_res.summary()
results = stat_res.results_df
results.head()

## 6. Significant genes

In [ ]:
significant = results[
    (results["padj"] < 0.05) & (results["log2FoldChange"].abs() >= 1)
]
upregulated = significant[significant["log2FoldChange"] > 0]
downregulated = significant[significant["log2FoldChange"] < 0]
print(f"Significant: {len(significant)} | Up: {len(upregulated)} | Down: {len(downregulated)}")

In [ ]:
top_up = upregulated.sort_values("log2FoldChange", ascending=False).head(20)
top_up

In [ ]:
top_down = downregulated.sort_values("log2FoldChange", ascending=True).head(20)
top_down

## 7. Volcano plot

In [ ]:
plot_df = results.dropna(subset=["padj"]).copy()
plot_df["-log10_padj"] = -np.log10(plot_df["padj"].clip(lower=1e-300))
plot_df["significant"] = (plot_df["padj"] < 0.05) & (plot_df["log2FoldChange"].abs() >= 1)

plt.figure(figsize=(10, 7))
ns = plot_df[~plot_df["significant"]]
sg = plot_df[plot_df["significant"]]
plt.scatter(ns["log2FoldChange"], ns["-log10_padj"], s=10, alpha=0.4, color="grey", label="Not significant")
plt.scatter(sg["log2FoldChange"], sg["-log10_padj"], s=12, alpha=0.7, color="crimson", label="Significant")

plt.axvline(1, linestyle="--", color="black", lw=0.8)
plt.axvline(-1, linestyle="--", color="black", lw=0.8)
plt.axhline(-np.log10(0.05), linestyle="--", color="black", lw=0.8)

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano Plot: Fibrosis vs Normal")
plt.legend()
plt.show()

## 8. PCA (على normalized counts)

In [ ]:
normed = dds.layers["normed_counts"]                     # samples x genes
log_counts = pd.DataFrame(
    np.log2(normed + 1),
    index=counts_for_dds.index,
    columns=counts_for_dds.columns,
)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(log_counts)

pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"], index=counts_for_dds.index)
pca_df["condition"] = metadata["condition"]

In [ ]:
plt.figure(figsize=(8, 6))
for condition in pca_df["condition"].cat.categories:
    subset = pca_df[pca_df["condition"] == condition]
    plt.scatter(subset["PC1"], subset["PC2"], label=condition, s=100)
    for sample, row in subset.iterrows():
        plt.annotate(sample, (row["PC1"], row["PC2"]), xytext=(5, 5), textcoords="offset points")

plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("PCA: Fibrosis vs Normal")
plt.legend()
plt.show()

## 9. Heatmap – top 50 DE genes\nنستخدم الـ normalized counts (وليس raw) ثم log2 ثم Z-score لكل جين. ترتيب العينات: fibrosis أولاً ثم normal.

In [ ]:
top50 = significant.sort_values("padj").head(50)

sample_order = (
    metadata.sort_values("condition", key=lambda s: s.cat.codes, ascending=False)
    .index.tolist()
)

heatmap_data = log_counts.loc[sample_order, top50.index].T       # genes x samples
heatmap_z = heatmap_data.sub(heatmap_data.mean(axis=1), axis=0).div(heatmap_data.std(axis=1), axis=0)

plt.figure(figsize=(10, 14))
plt.imshow(heatmap_z, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
plt.colorbar(label="Z-score")
plt.xticks(range(len(heatmap_z.columns)), heatmap_z.columns, rotation=45, ha="right")
plt.yticks(range(len(heatmap_z.index)), heatmap_z.index)
plt.xlabel("Samples")
plt.ylabel("Genes")
plt.title("Top 50 Differentially Expressed Genes")
plt.tight_layout()
plt.show()

## 10. Gene annotation (Ensembl → Symbol)\nنعمل annotation لكل الجينات المختبرة (وليس الدالة فقط) لأن GSEA يحتاج ترتيب كل الجينات.

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

res_all = results.dropna(subset=["stat"]).copy()
res_all["gene_id"] = res_all.index
res_all["query"] = res_all["gene_id"].astype(str).str.split(".").str[0]   # إزالة version مثل .1

annotations = mg.querymany(
    res_all["query"].unique().tolist(),
    scopes="ensembl.gene",
    fields="symbol,name",
    species="mouse",
    verbose=False,
)
annotation_df = pd.DataFrame(annotations)
if "notfound" in annotation_df.columns:
    annotation_df = annotation_df[annotation_df["notfound"].isna()]
annotation_df = annotation_df.drop_duplicates("query")[["query", "symbol", "name"]]

results_all_annotated = res_all.merge(annotation_df, on="query", how="left")
results_annotated = results_all_annotated[
    (results_all_annotated["padj"] < 0.05) & (results_all_annotated["log2FoldChange"].abs() >= 1)
].copy()

print("Annotated significant genes:", results_annotated["symbol"].notna().sum(), "/", len(results_annotated))
results_annotated[["gene_id", "symbol", "name", "log2FoldChange", "padj"]].head(10)

## 11. Enrichment analysis (GO & KEGG)

In [ ]:
import gseapy as gp

up_symbols = results_annotated.loc[results_annotated["log2FoldChange"] > 0, "symbol"].dropna().unique().tolist()
down_symbols = results_annotated.loc[results_annotated["log2FoldChange"] < 0, "symbol"].dropna().unique().tolist()
print("Upregulated genes:", len(up_symbols), "| Downregulated genes:", len(down_symbols))

COLS = ["Term", "Overlap", "P-value", "Adjusted P-value", "Genes"]

def run_enrichr(symbols, library):
    return gp.enrichr(gene_list=symbols, gene_sets=[library], organism="mouse", outdir=None)

In [ ]:
go_up = run_enrichr(up_symbols, "GO_Biological_Process_2023")
go_up.results[COLS].head(20)

In [ ]:
go_down = run_enrichr(down_symbols, "GO_Biological_Process_2023")
go_down.results[COLS].head(20)

In [ ]:
kegg_up = run_enrichr(up_symbols, "KEGG_2019_Mouse")
kegg_up.results[COLS].head(20)

In [ ]:
kegg_down = run_enrichr(down_symbols, "KEGG_2019_Mouse")
kegg_down.results[COLS].head(20)

## 12. GSEA (prerank على KEGG)\nالترتيب يعتمد على `stat` لكل الجينات المختبرة.

In [ ]:
results_gsea = results_all_annotated.dropna(subset=["symbol", "stat"]).drop_duplicates("symbol")
gene_ranking = results_gsea.set_index("symbol")["stat"].sort_values(ascending=False)
print("Genes for GSEA:", len(gene_ranking))
gene_ranking.head()

In [ ]:
gsea_kegg = gp.prerank(
    rnk=gene_ranking,
    gene_sets="KEGG_2019_Mouse",
    permutation_num=1000,
    outdir=None,
    seed=42,
)
gsea_kegg.res2d[["Term", "NES", "NOM p-val", "FDR q-val"]].head(15)

In [ ]:
TERM = "Oxidative phosphorylation"
terms = gsea_kegg.res2d["Term"].tolist()
print([t for t in terms if TERM in t])

In [ ]:
from gseapy.plot import gseaplot

ox = gsea_kegg.results[TERM]
gseaplot(
    rank_metric=gsea_kegg.ranking,
    term=TERM,
    hits=ox["hits"],
    nes=ox["nes"],
    pval=ox["pval"],
    fdr=ox["fdr"],
    RES=ox["RES"],
)
plt.show()

## 13. Save results

In [ ]:
results.to_csv("deseq2_results_all.csv")
results_annotated.to_csv("deseq2_significant_annotated.csv", index=False)
go_up.results.to_csv("go_up.csv", index=False)
go_down.results.to_csv("go_down.csv", index=False)
kegg_up.results.to_csv("kegg_up.csv", index=False)
kegg_down.results.to_csv("kegg_down.csv", index=False)
gsea_kegg.res2d.to_csv("gsea_kegg.csv", index=False)